In [50]:
from mne.io import read_raw_ctf 
from mne import read_epochs_fieldtrip,write_source_spaces
import numpy as np
from mne import pick_types,find_events,Epochs, EpochsArray, make_forward_solution
from mne.datasets import fetch_infant_template
import mne
from mne.beamformer import apply_lcmv, make_lcmv,apply_lcmv_epochs
import h5py
import scipy

In [29]:
file_name=['S04_heshengibp_20181029_01.ds',
'S05_heshengibp_20181029_05.ds',
'S06_heshengibp_20181122_01.ds',
'S07_heshengibp_20181122_03.ds',
'S08_heshengibp_20181123_01.ds',
'S09_heshengibp_20181128_01.ds',
'S10_heshengibp_20181129_02.ds',
'S01_heshengibp_20191018_01.ds',
'S02_heshengibp_20191018_01.ds',
'S04_heshengibp_20191023_01.ds',
'S05_heshengibp_20191023_01.ds',
'S06_heshengibp_20191023_01.ds',
'S07_heshengibp_20191023_01.ds',
'S08_heshengibp_20191023_01.ds',
'S09_heshengibp_20191023_01.ds',
'S10_heshengibp_20191024_01.ds',
'S11_heshengibp_20191028_01.ds',
'S12_heshengibp_20191028_05.ds',
'S13_heshengibp_20191030_01.ds',
'S14_heshengibp_20191030_01.ds',
'S15_heshengibp_20191030_01.ds']

In [30]:
file_name[0]

'S04_heshengibp_20181029_01.ds'

In [4]:
#for subj in range(21):
for subj in [7]:    

    data_path=r'F:\FXX_meg\\'+ file_name[subj]
    fname=r'F:\Moony_face_MEG_fxx\data_s'+ str(subj+1)+ '_ctfformate.mat'
    original_data = read_raw_ctf(data_path, preload=True)
    original_data.resample(100)
    data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney') 
    data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal') 
    events = find_events(original_data, stim_channel='UPPT001',shortest_event=1)
    #1 event_id=1 monney 2 normal
    epoch_monney = Epochs(original_data, events, event_id=1,tmin=-1, tmax= 2.499 ,preload=True,baseline=None)
    epoch_normal = Epochs(original_data, events, event_id=2,tmin=-1, tmax= 2.499 ,preload=True,baseline=None)
    def find_positions_with_M(strings):
        # 使用列表解析和enumerate来查找包含'M'的字符串及其位置信息
        positions = [index for index, string in enumerate(strings) if 'M' in string]
        return positions
    simulated_data_monney=epoch_monney.get_data(copy= False);
    simulated_data_monney=mne.baseline.rescale(simulated_data_monney,times=np.linspace(-1, 2.499, 100),baseline=[-0.2,0])
    simulated_data_monney[:,find_positions_with_M(epoch_monney.ch_names),:]=data_monney.get_data(copy= False);
    
    
    simulated_data_normal=epoch_normal.get_data(copy= False);
    simulated_data_normal=mne.baseline.rescale(simulated_data_normal,times=np.linspace(-1, 2.499, 100),baseline=[-0.2,0])
    simulated_data_normal[:,find_positions_with_M(epoch_normal.ch_names),:]=data_normal.get_data(copy= False);
    
    simulated_data_monney_epochs = EpochsArray(simulated_data_monney, epoch_monney.info)
    simulated_data_normal_epochs = EpochsArray(simulated_data_normal, epoch_normal.info)

    simulated_data_monney_epochs.pick('mag')
    simulated_data_normal_epochs.pick('mag')

    simulated_data_monney_epochs.filter(l_freq=2,h_freq=80)
    simulated_data_normal_epochs.filter(l_freq=2,h_freq=80)
    
    epochs_standard = simulated_data_monney_epochs.load_data()
    evoked_std_monney=epochs_standard.average()
    epochs_standard = simulated_data_normal_epochs.load_data()
    evoked_std_normal=epochs_standard.average()
    #del simulated_data_normal, simulated_data_monney, data_monney, data_normal, epoch_monney, epoch_normal
    trans = "fsaverage" 
    #src = fs_dir+"\bem"+"\fsaverage-ico-5-src.fif"
    src=r'C:\Users\jiangyong\mne_data\MNE-fsaverage-data\fsaverage\bem\fsaverage-ico-5-src.fif'
    #bem = fs_dir+"\bem"+ "\fsaverage-5120-5120-5120-bem-sol.fif"
    bem = r'C:\Users\jiangyong\mne_data\MNE-fsaverage-data\fsaverage\bem\fsaverage-5120-5120-5120-bem-sol.fif'


    fwd_monney =make_forward_solution(
        simulated_data_monney_epochs.info, trans=trans, src=src, bem=bem, meg=True, mindist=5.0, n_jobs=None
    )
    data_cov = mne.compute_covariance(simulated_data_monney_epochs, tmin=0.1+1, tmax=0.35+1, method="empirical")
    noise = mne.compute_covariance(simulated_data_monney_epochs,tmin=-0.25+1, tmax=0+1, method="empirical")
    filters = make_lcmv(
        evoked_std_monney.info,
        fwd_monney,
        data_cov,
        reg=0.01,
        noise_cov=noise,
        pick_ori="max-power",
        weight_norm="unit-noise-gain",
        rank=None,
    )
    # save a bit of memory
    src_monney = fwd_monney["src"]
    del fwd_monney
    stc_monney = apply_lcmv(evoked_std_monney, filters)
    fwd_normal =make_forward_solution(
    
        
        simulated_data_normal_epochs.info, trans=trans, src=src, bem=bem, meg=True, mindist=5.0, n_jobs=None
    )
    data_cov = mne.compute_covariance(simulated_data_normal_epochs, tmin=0.1+1, tmax=0.35+1, method="empirical")
    noise = mne.compute_covariance(simulated_data_normal_epochs,tmin=-0.25+1, tmax=0+1, method="empirical")
    filters = make_lcmv(
        evoked_std_normal.info,
        fwd_normal,
        data_cov,
        reg=0.01,
        noise_cov=noise,
        pick_ori="max-power",
        weight_norm="unit-noise-gain",
        rank=None,
    )
    # save a bit of memory
    src_normal = fwd_normal["src"]
    del fwd_normal
    stc_normal = apply_lcmv(evoked_std_normal, filters)
    saved_file_name1=r'F:\Moony_face_MEG_fxx\source_fsaveraged_s' + str(subj+1) +'monney.mat'
    saved_file_name2=r'F:\Moony_face_MEG_fxx\source_fsaveraged_s' + str(subj+1) +'normal.mat'
    scipy.io.savemat(saved_file_name1,{'monney_data':stc_monney.data})
    scipy.io.savemat(saved_file_name2,{'normal_data':stc_normal.data})

ds directory : F:\FXX_meg\S01_heshengibp_20191018_01.ds
    res4 data read.
    hc data read.
    Separate EEG position data file read.
    Quaternion matching (desired vs. transformed):
       1.70   75.71    0.00 mm <->    1.70   75.71    0.00 mm (orig :  -53.63   50.30 -253.23 mm) diff =    0.000 mm
      -1.70  -75.71    0.00 mm <->   -1.70  -75.71    0.00 mm (orig :   50.18  -59.97 -252.06 mm) diff =    0.000 mm
      97.18    0.00    0.00 mm <->   97.18    0.00    0.00 mm (orig :   58.30   55.17 -205.31 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
    Measurement info composed.
Finding samples for F:\FXX_meg\S01_heshengibp_20191018_01.ds\S01_heshengibp_20191018_01.meg4: 
    System clock channel is available, checking which samples are valid.
    1 x 696000 = 696000 samples from 409 chs
Current compensation grade : 3
Reading 0 ... 695999  =      0.000 ...   579.999 

C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\798252243.py:8: RuntimeWarning: Importing FieldTrip data without an info dict from the original file. Channel locations, orientations and types will be incorrect. The imported data cannot be used for source analysis, channel interpolation etc.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\798252243.py:8: RuntimeWarning: Cannot guess the correct type of channel MSTAT101. Making it a MISC channel.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\798252243.py:8: RuntimeWarning: Cannot guess the correct type of channel MRSYN101. Making it a MISC channel.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\798252243.py:8: RuntimeWarning: Cannot guess the correct type of channel 

Adding metadata with 1 columns
120 matching events found
No baseline correction applied


C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\798252243.py:9: RuntimeWarning: Importing FieldTrip data without an info dict from the original file. Channel locations, orientations and types will be incorrect. The imported data cannot be used for source analysis, channel interpolation etc.
  data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\798252243.py:9: RuntimeWarning: Cannot guess the correct type of channel MSTAT101. Making it a MISC channel.
  data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\798252243.py:9: RuntimeWarning: Cannot guess the correct type of channel MRSYN101. Making it a MISC channel.
  data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\798252243.py:9: RuntimeWarning: Cannot guess the correct type of channel 

Adding metadata with 1 columns
119 matching events found
No baseline correction applied
Trigger channel UPPT001 has a non-zero initial value of {initial_value} (consider using initial_event=True to detect this event)
1427 events found on stim channel UPPT001
Event IDs: [1 2 4]
Not setting metadata
644 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 644 events and 351 original time points ...
1 bad epochs dropped
Not setting metadata
552 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 552 events and 351 original time points ...
5 bad epochs dropped
Applying baseline correction (mode: mean)


ValueError: shape mismatch: value array of shape (120,348,4200) could not be broadcast to indexing result of shape (643,348,351)

In [5]:
subj=7    

data_path=r'E:\FXX_meg\\'+ file_name[subj]
fname=r'E:\Moony_face_MEG_fxx\data_s'+ str(subj+1)+ '_ctfformate.mat'
original_data = read_raw_ctf(data_path, preload=True)



ds directory : E:\FXX_meg\S01_heshengibp_20191018_01.ds
    res4 data read.
    hc data read.
    Separate EEG position data file read.
    Quaternion matching (desired vs. transformed):
       1.70   75.71    0.00 mm <->    1.70   75.71    0.00 mm (orig :  -53.63   50.30 -253.23 mm) diff =    0.000 mm
      -1.70  -75.71    0.00 mm <->   -1.70  -75.71    0.00 mm (orig :   50.18  -59.97 -252.06 mm) diff =    0.000 mm
      97.18    0.00    0.00 mm <->   97.18    0.00    0.00 mm (orig :   58.30   55.17 -205.31 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
    Measurement info composed.
Finding samples for E:\FXX_meg\S01_heshengibp_20191018_01.ds\S01_heshengibp_20191018_01.meg4: 
    System clock channel is available, checking which samples are valid.
    1 x 696000 = 696000 samples from 409 chs
Current compensation grade : 3
Reading 0 ... 695999  =      0.000 ...   579.999 

In [6]:
data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney') 

C:\Users\jiangyong\AppData\Local\Temp\ipykernel_32468\1539251285.py:1: RuntimeWarning: Importing FieldTrip data without an info dict from the original file. Channel locations, orientations and types will be incorrect. The imported data cannot be used for source analysis, channel interpolation etc.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_32468\1539251285.py:1: RuntimeWarning: Cannot guess the correct type of channel MSTAT101. Making it a MISC channel.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_32468\1539251285.py:1: RuntimeWarning: Cannot guess the correct type of channel MRSYN101. Making it a MISC channel.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_32468\1539251285.py:1: RuntimeWarning: Cannot guess the correct type of chan

Not setting metadata
120 matching events found
No baseline correction applied


C:\Users\jiangyong\AppData\Local\Temp\ipykernel_32468\1539251285.py:1: RuntimeWarning: The Pandas library is not installed. Not returning the original trialinfo matrix as metadata.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')


In [8]:
np.shape(original_data.get_data())

(409, 696000)

In [9]:
events = find_events(original_data, stim_channel='UPPT001',shortest_event=1)
#1 event_id=1 monney 2 normal
epoch_monney = Epochs(original_data, events, event_id=1,tmin=-1, tmax= 2.499 ,preload=True,baseline=None)

Trigger channel UPPT001 has a non-zero initial value of {initial_value} (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
240 events found on stim channel UPPT001
Event IDs: [1 2]
Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 120 events and 4200 original time points ...
0 bad epochs dropped


In [15]:
np.shape(data_monney.get_data())

C:\Users\jiangyong\AppData\Local\Temp\ipykernel_32468\3324909728.py:1: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  np.shape(data_monney.get_data())


(120, 348, 4200)

In [11]:
epoch_monney.ch_names

['SCLK01-177',
 'BG1-4503',
 'BG2-4503',
 'BG3-4503',
 'BP1-4503',
 'BP2-4503',
 'BP3-4503',
 'BR1-4503',
 'BR2-4503',
 'BR3-4503',
 'G11-4503',
 'G12-4503',
 'G13-4503',
 'G22-4503',
 'G23-4503',
 'P11-4503',
 'P12-4503',
 'P13-4503',
 'P22-4503',
 'P23-4503',
 'Q11-4503',
 'Q12-4503',
 'Q13-4503',
 'Q22-4503',
 'Q23-4503',
 'R11-4503',
 'R12-4503',
 'R13-4503',
 'R22-4503',
 'R23-4503',
 'MLC11-4503',
 'MLC12-4503',
 'MLC13-4503',
 'MLC14-4503',
 'MLC15-4503',
 'MLC16-4503',
 'MLC17-4503',
 'MLC21-4503',
 'MLC22-4503',
 'MLC23-4503',
 'MLC24-4503',
 'MLC25-4503',
 'MLC31-4503',
 'MLC32-4503',
 'MLC41-4503',
 'MLC42-4503',
 'MLC51-4503',
 'MLC52-4503',
 'MLC53-4503',
 'MLC54-4503',
 'MLC55-4503',
 'MLC61-4503',
 'MLC62-4503',
 'MLC63-4503',
 'MLF11-4503',
 'MLF12-4503',
 'MLF13-4503',
 'MLF14-4503',
 'MLF21-4503',
 'MLF22-4503',
 'MLF23-4503',
 'MLF24-4503',
 'MLF25-4503',
 'MLF31-4503',
 'MLF32-4503',
 'MLF33-4503',
 'MLF34-4503',
 'MLF35-4503',
 'MLF41-4503',
 'MLF42-4503',
 'MLF43-

In [13]:
def find_positions_with_M(strings):
        # 使用列表解析和enumerate来查找包含'M'的字符串及其位置信息
    positions = [index for index, string in enumerate(strings) if 'M' in string]
    return positions
find_positions_with_M(epoch_monney.ch_names)

[30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 207,
 2

In [16]:
    def find_positions_with_M(strings):
        # 使用列表解析和enumerate来查找包含'M'的字符串及其位置信息
        positions = [index for index, string in enumerate(strings) if 'M' in string]
        return positions
    simulated_data_monney=epoch_monney.get_data(copy= False);
    simulated_data_monney=mne.baseline.rescale(simulated_data_monney,times=np.linspace(-1, 2.499, 1200),baseline=[-0.2,0])
    simulated_data_monney[:,find_positions_with_M(epoch_monney.ch_names),:]=data_monney.get_data(copy= False);
    
    simulated_data_monney_epochs = EpochsArray(simulated_data_monney, epoch_monney.info)

    simulated_data_monney_epochs.filter(l_freq=2,h_freq=80)

Applying baseline correction (mode: mean)
Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated
Setting up band-pass filter from 2 - 80 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 2.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 1.00 Hz)
- Upper passband edge: 80.00 Hz
- Upper transition bandwidth: 20.00 Hz (-6 dB cutoff frequency: 90.00 Hz)
- Filter length: 1981 samples (1.651 s)



Number of events,120
Events,1: 120
Time range,0.000 – 3.499 s
Baseline,off


In [41]:
simulated_data_monney_epochs = EpochsArray(simulated_data_monney, epoch_monney.info)

Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated


In [43]:
np.shape(simulated_data_monney_epochs.get_data(copy=False))

(120, 272, 4200)

In [42]:
simulated_data_monney_epochs.pick('mag')

Removing 5 compensators from info because not all compensation channels were picked.


Number of events,120
Events,1: 120
Time range,0.000 – 3.499 s
Baseline,off


In [ ]:
#for subj in range(21):
for subj in [7]:    

    data_path=r'F:\FXX_meg\\'+ file_name[subj]
    fname=r'F:\Moony_face_MEG_fxx\data_s'+ str(subj+1)+ '_ctfformate.mat'
    original_data = read_raw_ctf(data_path, preload=True)
    data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney') 
    data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal') 
    events = find_events(original_data, stim_channel='UPPT001',shortest_event=1)
    #1 event_id=1 monney 2 normal
    epoch_monney = Epochs(original_data, events, event_id=1,tmin=-1, tmax= 2.499 ,preload=True,baseline=None)
    epoch_normal = Epochs(original_data, events, event_id=2,tmin=-1, tmax= 2.499 ,preload=True,baseline=None)
    def find_positions_with_M(strings):
        # 使用列表解析和enumerate来查找包含'M'的字符串及其位置信息
        positions = [index for index, string in enumerate(strings) if 'M' in string]
        return positions
    simulated_data_monney=epoch_monney.get_data(copy= False);
    simulated_data_monney=mne.baseline.rescale(simulated_data_monney,times=np.linspace(-1, 2.499, 1200),baseline=[-0.2,0])
    simulated_data_monney[:,find_positions_with_M(epoch_monney.ch_names),:]=data_monney.get_data(copy= False);
    
    
    simulated_data_normal=epoch_normal.get_data(copy= False);
    simulated_data_normal=mne.baseline.rescale(simulated_data_normal,times=np.linspace(-1, 2.499, 1200),baseline=[-0.2,0])
    simulated_data_normal[:,find_positions_with_M(epoch_normal.ch_names),:]=data_normal.get_data(copy= False);
    
    simulated_data_monney_epochs = EpochsArray(simulated_data_monney, epoch_monney.info)
    simulated_data_normal_epochs = EpochsArray(simulated_data_normal, epoch_normal.info)

    simulated_data_monney_epochs.pick('mag')
    simulated_data_normal_epochs.pick('mag')

    simulated_data_monney_epochs.filter(l_freq=2,h_freq=80)
    simulated_data_normal_epochs.filter(l_freq=2,h_freq=80)
    
    epochs_standard = simulated_data_monney_epochs.load_data()
    evoked_std_monney=epochs_standard
    epochs_standard = simulated_data_normal_epochs.load_data()
    evoked_std_normal=epochs_standard
    #del simulated_data_normal, simulated_data_monney, data_monney, data_normal, epoch_monney, epoch_normal
    trans = "fsaverage" 
    #src = fs_dir+"\bem"+"\fsaverage-ico-5-src.fif"
    src=r'C:\Users\jiangyong\mne_data\MNE-fsaverage-data\fsaverage\bem\fsaverage-ico-5-src.fif'
    #bem = fs_dir+"\bem"+ "\fsaverage-5120-5120-5120-bem-sol.fif"
    bem = r'C:\Users\jiangyong\mne_data\MNE-fsaverage-data\fsaverage\bem\fsaverage-5120-5120-5120-bem-sol.fif'


    fwd_monney =make_forward_solution(
        simulated_data_monney_epochs.info, trans=trans, src=src, bem=bem, meg=True, mindist=5.0, n_jobs=None
    )
    data_cov = mne.compute_covariance(simulated_data_monney_epochs, tmin=0.1+1, tmax=0.35+1, method="empirical")
    noise = mne.compute_covariance(simulated_data_monney_epochs,tmin=-0.25+1, tmax=0+1, method="empirical")
    filters = make_lcmv(
        evoked_std_monney.info,
        fwd_monney,
        data_cov,
        reg=0.01,
        noise_cov=noise,
        pick_ori="max-power",
        weight_norm="unit-noise-gain",
        rank=None,
    )
    # save a bit of memory
    src_monney = fwd_monney["src"]
    del fwd_monney
    stc_monney = apply_lcmv_epochs(evoked_std_monney, filters)
    fwd_normal =make_forward_solution(
    
        
        simulated_data_normal_epochs.info, trans=trans, src=src, bem=bem, meg=True, mindist=5.0, n_jobs=None
    )
    data_cov = mne.compute_covariance(simulated_data_normal_epochs, tmin=0.1+1, tmax=0.35+1, method="empirical")
    noise = mne.compute_covariance(simulated_data_normal_epochs,tmin=-0.25+1, tmax=0+1, method="empirical")
    filters = make_lcmv(
        evoked_std_normal.info,
        fwd_normal,
        data_cov,
        reg=0.01,
        noise_cov=noise,
        pick_ori="max-power",
        weight_norm="unit-noise-gain",
        rank=None,
    )
    # save a bit of memory
    src_normal = fwd_normal["src"]
    del fwd_normal
    stc_normal = apply_lcmv_epochs(evoked_std_normal, filters)
    #saved_file_name1=r'E:\Moony_face_MEG_fxx\source_fsaveraged_s' + str(subj+1) +'monney.mat'
    #saved_file_name2=r'E:\Moony_face_MEG_fxx\source_fsaveraged_s' + str(subj+1) +'normal.mat'
    #scipy.io.savemat(saved_file_name1,{'monney_data':stc_monney.data})
    #scipy.io.savemat(saved_file_name2,{'normal_data':stc_normal.data})

ds directory : F:\FXX_meg\S01_heshengibp_20191018_01.ds
    res4 data read.
    hc data read.
    Separate EEG position data file read.
    Quaternion matching (desired vs. transformed):
       1.70   75.71    0.00 mm <->    1.70   75.71    0.00 mm (orig :  -53.63   50.30 -253.23 mm) diff =    0.000 mm
      -1.70  -75.71    0.00 mm <->   -1.70  -75.71    0.00 mm (orig :   50.18  -59.97 -252.06 mm) diff =    0.000 mm
      97.18    0.00    0.00 mm <->   97.18    0.00    0.00 mm (orig :   58.30   55.17 -205.31 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
    Measurement info composed.
Finding samples for F:\FXX_meg\S01_heshengibp_20191018_01.ds\S01_heshengibp_20191018_01.meg4: 
    System clock channel is available, checking which samples are valid.
    1 x 696000 = 696000 samples from 409 chs
Current compensation grade : 3
Reading 0 ... 695999  =      0.000 ...   579.999 

C:\Users\jiangyong\AppData\Local\Temp\ipykernel_19048\3537457965.py:7: RuntimeWarning: Importing FieldTrip data without an info dict from the original file. Channel locations, orientations and types will be incorrect. The imported data cannot be used for source analysis, channel interpolation etc.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_19048\3537457965.py:7: RuntimeWarning: Cannot guess the correct type of channel MSTAT101. Making it a MISC channel.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_19048\3537457965.py:7: RuntimeWarning: Cannot guess the correct type of channel MRSYN101. Making it a MISC channel.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_19048\3537457965.py:7: RuntimeWarning: Cannot guess the correct type of chan

Adding metadata with 1 columns
120 matching events found
No baseline correction applied


C:\Users\jiangyong\AppData\Local\Temp\ipykernel_19048\3537457965.py:8: RuntimeWarning: Importing FieldTrip data without an info dict from the original file. Channel locations, orientations and types will be incorrect. The imported data cannot be used for source analysis, channel interpolation etc.
  data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_19048\3537457965.py:8: RuntimeWarning: Cannot guess the correct type of channel MSTAT101. Making it a MISC channel.
  data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_19048\3537457965.py:8: RuntimeWarning: Cannot guess the correct type of channel MRSYN101. Making it a MISC channel.
  data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_19048\3537457965.py:8: RuntimeWarning: Cannot guess the correct type of chan

Adding metadata with 1 columns
119 matching events found
No baseline correction applied
Trigger channel UPPT001 has a non-zero initial value of {initial_value} (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
240 events found on stim channel UPPT001
Event IDs: [1 2]
Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 120 events and 4200 original time points ...
0 bad epochs dropped
Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 120 events and 4200 original time points ...
1 bad epochs dropped
Applying baseline correction (mode: mean)
Applying baseline correction (mode: mean)
Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
119 matching events found
No baseline correc

In [14]:
epochs_standard

Number of events,119
Events,1: 119
Time range,0.000 – 3.499 s
Baseline,off


In [9]:
evoked_std_monney

Number of events,120
Events,1: 120
Time range,0.000 – 3.499 s
Baseline,off


In [79]:
    data_path=r'F:\FXX_meg\\'+ file_name[subj]
    fname=r'F:\Moony_face_MEG_fxx\data_s'+ str(subj+1)+ '_ctfformate_resample100.mat'
    original_data = read_raw_ctf(data_path, preload=True)
    #original_data.resample(100)
    data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney') 
    data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal') 
    events = find_events(original_data, stim_channel='UPPT001',shortest_event=1)
    #1 event_id=1 monney 2 normal
    epoch_monney = Epochs(original_data, events, event_id=1,tmin=-1, tmax= 2.50 ,preload=True,baseline=None)
    epoch_monney.resample(100)
    epoch_normal = Epochs(original_data, events, event_id=2,tmin=-1, tmax= 2.50 ,preload=True,baseline=None)
    epoch_normal.resample(100)
    def find_positions_with_M(strings):
        # 使用列表解析和enumerate来查找包含'M'的字符串及其位置信息
        positions = [index for index, string in enumerate(strings) if 'M' in string]
        return positions
    simulated_data_monney=epoch_monney.get_data(copy= False);
    simulated_data_monney=mne.baseline.rescale(simulated_data_monney,times=np.linspace(-1, 2.50, 100),baseline=[-0.2,0])
    simulated_data_monney[:,find_positions_with_M(epoch_monney.ch_names),:]=data_monney.get_data(copy= False);
    
    
    simulated_data_normal=epoch_normal.get_data(copy= False);
    simulated_data_normal=mne.baseline.rescale(simulated_data_normal,times=np.linspace(-1, 2.50, 100),baseline=[-0.2,0])
    simulated_data_normal[:,find_positions_with_M(epoch_normal.ch_names),:]=data_normal.get_data(copy= False);
    
    simulated_data_monney_epochs = EpochsArray(simulated_data_monney, epoch_monney.info)
    simulated_data_normal_epochs = EpochsArray(simulated_data_normal, epoch_normal.info)

    simulated_data_monney_epochs.pick('mag')
    simulated_data_normal_epochs.pick('mag')

    simulated_data_monney_epochs.filter(l_freq=2,h_freq=40)
    simulated_data_normal_epochs.filter(l_freq=2,h_freq=40)
    
    epochs_standard = simulated_data_monney_epochs.load_data()
    evoked_std_monney=epochs_standard
    epochs_standard = simulated_data_normal_epochs.load_data()
    evoked_std_normal=epochs_standard
    #del simulated_data_normal, simulated_data_monney, data_monney, data_normal, epoch_monney, epoch_normal
    trans = "fsaverage" 
    #src = fs_dir+"\bem"+"\fsaverage-ico-5-src.fif"
    src=r'C:\Users\jiangyong\mne_data\MNE-fsaverage-data\fsaverage\bem\fsaverage-ico-5-src.fif'
    #bem = fs_dir+"\bem"+ "\fsaverage-5120-5120-5120-bem-sol.fif"
    bem = r'C:\Users\jiangyong\mne_data\MNE-fsaverage-data\fsaverage\bem\fsaverage-5120-5120-5120-bem-sol.fif'


    fwd_monney =make_forward_solution(
        simulated_data_monney_epochs.info, trans=trans, src=src, bem=bem, meg=True, mindist=5.0, n_jobs=None
    )
    data_cov = mne.compute_covariance(simulated_data_monney_epochs, tmin=0.1+1, tmax=0.35+1, method="empirical")
    noise = mne.compute_covariance(simulated_data_monney_epochs,tmin=-0.25+1, tmax=0+1, method="empirical")
    filters = make_lcmv(
        evoked_std_monney.info,
        fwd_monney,
        data_cov,
        reg=0.01,
        noise_cov=noise,
        pick_ori="max-power",
        weight_norm="unit-noise-gain",
        rank=None,
    )
    # save a bit of memory
    src_monney = fwd_monney["src"]
    del fwd_monney
    stc_monney = apply_lcmv_epochs(evoked_std_monney, filters)
    fwd_normal =make_forward_solution(
    
        
        simulated_data_normal_epochs.info, trans=trans, src=src, bem=bem, meg=True, mindist=5.0, n_jobs=None
    )
    data_cov = mne.compute_covariance(simulated_data_normal_epochs, tmin=0.1+1, tmax=0.35+1, method="empirical")
    noise = mne.compute_covariance(simulated_data_normal_epochs,tmin=-0.25+1, tmax=0+1, method="empirical")
    filters = make_lcmv(
        evoked_std_normal.info,
        fwd_normal,
        data_cov,
        reg=0.01,
        noise_cov=noise,
        pick_ori="max-power",
        weight_norm="unit-noise-gain",
        rank=None,
    )
    # save a bit of memory
    src_normal = fwd_normal["src"]
    del fwd_normal
    stc_normal = apply_lcmv_epochs(evoked_std_normal, filters)




    stcrh_file_name='s'+str(subj+1)+'monney_100_surfstc-rh.stc'
    stclh_file_name='s'+str(subj+1)+'monney_100_surfstc-lh.stc'
    src_file_name='s'+str(subj+1)+'monney_100_surf.fif'
    stc_monney.save(src_file_name,ftype='stc')
    write_source_spaces(src_file_name, src_monney, overwrite=True)


    stcrh_file_name='s'+str(subj+1)+'normal_100_surfstc-rh.stc'
    stclh_file_name='s'+str(subj+1)+'normal_100_surfstc-lh.stc'
    src_file_name='s'+str(subj+1)+'normal_100_surf.fif'
    stc_normal.save(src_file_name,ftype='stc')
    write_source_spaces(src_file_name, src_normal, overwrite=True)

ds directory : F:\FXX_meg\S01_heshengibp_20191018_01.ds
    res4 data read.
    hc data read.
    Separate EEG position data file read.
    Quaternion matching (desired vs. transformed):
       1.70   75.71    0.00 mm <->    1.70   75.71    0.00 mm (orig :  -53.63   50.30 -253.23 mm) diff =    0.000 mm
      -1.70  -75.71    0.00 mm <->   -1.70  -75.71    0.00 mm (orig :   50.18  -59.97 -252.06 mm) diff =    0.000 mm
      97.18    0.00    0.00 mm <->   97.18    0.00    0.00 mm (orig :   58.30   55.17 -205.31 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
    Measurement info composed.
Finding samples for F:\FXX_meg\S01_heshengibp_20191018_01.ds\S01_heshengibp_20191018_01.meg4: 
    System clock channel is available, checking which samples are valid.
    1 x 696000 = 696000 samples from 409 chs
Current compensation grade : 3
Reading 0 ... 695999  =      0.000 ...   579.999 

C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\3898146721.py:5: RuntimeWarning: Importing FieldTrip data without an info dict from the original file. Channel locations, orientations and types will be incorrect. The imported data cannot be used for source analysis, channel interpolation etc.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\3898146721.py:5: RuntimeWarning: Cannot guess the correct type of channel MSTAT101. Making it a MISC channel.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\3898146721.py:5: RuntimeWarning: Cannot guess the correct type of channel MRSYN101. Making it a MISC channel.
  data_monney = read_epochs_fieldtrip(fname, info = None, data_name='data_monney')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\3898146721.py:5: RuntimeWarning: Cannot guess the correct type of chan

Adding metadata with 1 columns
119 matching events found
No baseline correction applied
Trigger channel UPPT001 has a non-zero initial value of {initial_value} (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
240 events found on stim channel UPPT001
Event IDs: [1 2]
Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 120 events and 4201 original time points ...


C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\3898146721.py:6: RuntimeWarning: Importing FieldTrip data without an info dict from the original file. Channel locations, orientations and types will be incorrect. The imported data cannot be used for source analysis, channel interpolation etc.
  data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\3898146721.py:6: RuntimeWarning: Cannot guess the correct type of channel MSTAT101. Making it a MISC channel.
  data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\3898146721.py:6: RuntimeWarning: Cannot guess the correct type of channel MRSYN101. Making it a MISC channel.
  data_normal = read_epochs_fieldtrip(fname, info = None, data_name='data_normal')
C:\Users\jiangyong\AppData\Local\Temp\ipykernel_14944\3898146721.py:6: RuntimeWarning: Cannot guess the correct type of chan

0 bad epochs dropped
Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 120 events and 4201 original time points ...
1 bad epochs dropped
Applying baseline correction (mode: mean)
Applying baseline correction (mode: mean)
Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
119 matching events found
No baseline correction applied
0 projection items activated
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Setting up band-pass filter from 2 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 2.00
- Lo

AttributeError: 'list' object has no attribute 'save'